# 03. Freeze the selected model and test on JECAM

The selected candidate is refitted on the training and validation partitions with 1,000 trees, frozen, then scored once on the primary JECAM test parcels.

**Input:** current selection. **Output:** `current_models.json`, `current_test.json`. **Next:** 04.

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # experiments package

import pandas as pd
from IPython.display import display
from experiments.improved_v1 import workflow
pd.set_option("display.width", 200)

## Freeze

In [ ]:
selection_handoff = ""  # Optional selection receipt; blank uses the current pointer
models = workflow.freeze(root, selection_handoff or None)
info = json.loads((models.parent / "model.json").read_text())
print("Models receipt:", models)
print("Candidate:", info["candidate"], "| predictors:", len(info["feature_columns"]), "| development parcels:", len(info["development_groups"]))

## Test

In [ ]:
tested = workflow.test(root)
scores = json.loads((tested.parent / "test_scores.json").read_text())
populations = scores["populations"]
display(pd.DataFrame([{"population": name, "parcels": r["parcels"], "crop_coverage": r["crop_coverage"],
                       "macro_f1": r["metrics"]["macro_f1"], "present_crop_macro_f1": r["metrics"]["macro_f1_present_crops"],
                       "majority_baseline": r["majority_baseline_macro_f1"], "meets_f1_criteria": r["meets_f1_criteria"],
                       **{f"{c}_f1": v["f1"] for c, v in r["metrics"]["per_crop"].items() if v["support"]},
                       **{f"{c}_parcels": v["support"] for c, v in r["metrics"]["per_crop"].items()}}
                      for name, r in populations.items()]).round(3))
for name, r in populations.items():
    m = r["metrics"]
    print(name)
    display(pd.DataFrame(m["confusion_matrix"], index=m["classes"], columns=m["classes"]))

## Main pipeline results for reference

In [ ]:
from crop_pipeline.analysis_workflow import resolve_stage_pointer
official = json.loads((resolve_stage_pointer(root / "pipeline_outputs/testing/current_scores.json").parent / "reports/test_scores.json").read_text())
display(pd.DataFrame([{"model": f"main pipeline {k}", "jecam_macro_f1": v.get("macro_f1"),
                       "parcels": sum(c["support"] for c in v["per_crop"].values())} for k, v in official.items() if "per_crop" in v] +
                     [{"model": f"improved {scores['candidate']}", "jecam_macro_f1": populations["jecam"]["metrics"]["macro_f1"],
                       "parcels": populations["jecam"]["parcels"]}]).round(3))